## Scale and Align Synthetic Samples to Historical Time Series

The functions in this code are for design-based appraoch. For response-based approach, simply ignore "for year in years:" in each function.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

In [ ]:
def sample_events(historical_values, target_value, min_diff=1e-3, power=1.5, seed=None):
    """
    Sample an index from historical values with probability inversely proportional to a powered distance.

    Parameters:
    - historical_values (array-like): Historical values (e.g., peaks)
    - target_value (float): The value to match
    - min_diff (float): Minimum difference to avoid division by zero
    - power (float): Exponent to control how aggressively distance is penalized
    - seed (int): Random seed for reproducibility

    Returns:
    - selected_index (int): Index of the selected sample
    - probabilities (np.ndarray): Normalized sampling probabilities
    """
    historical_values = np.array(historical_values)
    differences = np.abs(historical_values - target_value)
    differences = np.maximum(differences, min_diff)
    
    weights = 1.0 / (differences ** power)  # <-- POWER used here
    probabilities = weights / weights.sum()

    rng = np.random.default_rng(seed)
    selected_index = rng.choice(len(historical_values), p=probabilities)
    return selected_index, probabilities



def scale_time_series(selected_curve, target_peak, curve_type="NTR"):
    """
    Scale the selected time series to match the target peak value.

    Parameters:
    - selected_curve (array-like): Time series of the historical event
    - target_peak (float): Target peak value to scale to
    - curve_type (str): Optional, "NTR" or "RF" for debugging/printing

    Returns:
    - scaled_curve (np.ndarray): Time series scaled to match the target peak
    """
    selected_curve = np.array(selected_curve)
    current_peak = np.max(selected_curve)
    scale_factor = target_peak / current_peak
    scaled_curve = np.array(selected_curve) * scale_factor
    return scaled_curve



In [ ]:
def scale_and_align_center_variable_one_station(center_variable_dataset_col_name, 
                            center_variable_pot_col_name,  
                            pot_time,
                            pot_timeseries_event_time, 
                            pot_timeseries_time, 
                            number_of_events_per_year, 
                            center_variable_pot_path,
                            center_variable_pot_timeseries_path, 
                            synthetic_samples_from_isolines_directory, 
                            center_variable_synthetic_col,
                            center_variable_output_path, 
                            center_variable_output_col, 
                            center_variable_output_scale_factor_col,
                            years):
        
    
    """
    Generates scaled time series for a center variable (single station case).

    Parameters
    ----------
    years : list of str
        Return periods (e.g., ['1','5','10','20']).

    synthetic_input_directory : str
        Directory containing synthetic bivariate samples.

    number_of_events_per_year : int
        Number of events used in POT analysis.

    pot_input_path : str
        Path to POT dataset for the center variable.

    timeseries_input_path : str
        Path to historical event time series around the POT.

    center_variable_dataset_col_name : str
        Column name of the center variable in the original dataset.

    center_variable_pot_col_name : str
        Column name of the center variable in the POT dataset.

    center_variable_output_path : str
        Directory where scaled outputs will be saved.

    pot_time : str
        Time column name in the POT dataset.

    pot_timeseries_event_time : str
        Event identifier column in time series dataset.

    pot_timeseries_time : str
        Time column in time series dataset.

    seed : int, optional
        Random seed for reproducibility.

    Returns
    -------
    None

    Notes
    -----
    For each synthetic event:
    1. Select a similar historical event using POT peaks.
    2. Extract the corresponding time series.
    3. Scale it to match the synthetic peak.
    4. Save the scaled event to file.

    This function is used for variables such as NTR or RF when they act as the center variable.
    """

    df_pot = pd.read_csv(center_variable_pot_path)
    df_pot['time'] = pd.to_datetime(df_pot[pot_time])
    historical_peaks = df_pot[center_variable_pot_col_name].values  

    df_timeseries_pot = pd.read_csv(center_variable_pot_timeseries_path)
    df_timeseries_pot['event_time'] = pd.to_datetime(df_timeseries_pot[pot_timeseries_event_time])
    df_timeseries_pot = df_timeseries_pot.sort_values(by=['event_time'])

    for year in years: # IF you are doing response-based analysis, you don't need the years. You only have one file containing the representative space.
        target_df = pd.read_csv(rf'{synthetic_samples_from_isolines_directory}/Synthetic_{year}yr_Bivariate_from_Trivariate_{number_of_events_per_year}.csv')
        for i in range(len(target_df)):
            target_center_variable_peak = target_df.loc[i,center_variable_synthetic_col]  
            # Sample and scale
            idx, probs = sample_events(historical_peaks, target_center_variable_peak, power=1.5, seed=42)
            print(f"Historical Peak: {historical_peaks[idx]}, Generated Peak: {target_center_variable_peak}, sampling probability: {probs[idx]:.3f}")
            historical_curves = df_timeseries_pot[df_timeseries_pot['event_time'] == df_pot['time'].iloc[idx]]
            selected_curve = historical_curves
            scaled_curve = scale_time_series(selected_curve[center_variable_dataset_col_name], target_center_variable_peak, curve_type="NTR")

            center_variable_scale_factor = target_center_variable_peak / selected_curve[center_variable_dataset_col_name].max()
            
            df_pot_center_variable_scaled = pd.DataFrame({
                'Time': pd.to_datetime(selected_curve[pot_timeseries_time].values),
                center_variable_output_col: scaled_curve,
                center_variable_output_scale_factor_col: center_variable_scale_factor
            })
            df_pot_center_variable_scaled['Time'] = pd.to_datetime(df_pot_center_variable_scaled['Time'])
    
            df_pot_center_variable_scaled = df_pot_center_variable_scaled.sort_values(by='Time')
            df_pot_center_variable_scaled.to_csv(center_variable_output_path, index=False)


In [ ]:
def scale_and_align_center_variable_two_stations(center_variable_dataset_col_name, 
                            center_variable_dataset_time, 
                            center_variable_dataset_path_station_1, 
                            center_variable_dataset_path_station_2,
                            center_variable_pot_col_name,  
                            pot_time,
                            pot_timeseries_event_time, 
                            pot_timeseries_time, 
                            number_of_events_per_year, 
                            center_variable_pot_path,
                            center_variable_pot_timeseries_path, 
                            synthetic_samples_from_isolines_directory, 
                            center_variable_synthetic_col,
                            center_variable_output_path_station_1, 
                            center_variable_output_path_station_2, 
                            center_variable_output_col, 
                            center_variable_output_scale_factor_col,
                            years):
    """
    Generates scaled time series for a center variable with two stations (e.g., RD at Biloxi and Pascagoula).

    Parameters
    ----------
        years : list of str
        Return periods (e.g., ['1','5','10','20']).

    synthetic_input_directory : str
        Directory containing synthetic bivariate samples.

    number_of_events_per_year : int
        Number of events used in POT analysis.

    pot_input_path : str
        Path to POT dataset for the center variable.

    timeseries_input_path : str
        Path to historical event time series around the POT.

    center_variable_dataset_col_name : str
        Column name of the center variable in the original dataset.

    center_variable_pot_col_name : str
        Column name of the center variable in the POT dataset.

    center_variable_output_path : str
        Directory where scaled outputs will be saved.

    pot_time : str
        Time column name in the POT dataset.

    pot_timeseries_event_time : str
        Event identifier column in time series dataset.

    pot_timeseries_time : str
        Time column in time series dataset.

    seed : int, optional
        Random seed for reproducibility.

    Returns
    -------
    None

    Notes
    -----
    This function is used when the center variable has multiple spatial locations.
    The scaling and event selection process is performed independently for each station.
    """


    df_pot = pd.read_csv(center_variable_pot_path)
    df_pot['time'] = pd.to_datetime(df_pot[pot_time])
    cutoff = pd.Timestamp('1987-11-10 23:59:59')
    df_pot = df_pot[df_pot['time'] > cutoff]
    historical_peaks = df_pot[center_variable_pot_col_name].values  

    df_timeseries_pot = pd.read_csv(center_variable_pot_timeseries_path)
    df_timeseries_pot['event_time'] = pd.to_datetime(df_timeseries_pot[pot_timeseries_event_time])
    df_timeseries_pot = df_timeseries_pot.sort_values(by=['event_time'])


    # ======================================================
    # Station 1
    # ======================================================
    df_original_timeseries = pd.read_csv(center_variable_dataset_path_station_1)
    df_original_timeseries['Time'] = pd.to_datetime(df_original_timeseries[center_variable_dataset_time], utc=True).dt.tz_localize(None)
    df_original_timeseries = df_original_timeseries.rename(columns={center_variable_dataset_col_name:'hourly_discharge_m3s'})
    df_original_timeseries = df_original_timeseries[['Time','hourly_discharge_m3s']]

    for year in years:
        target_df = pd.read_csv(rf'{synthetic_samples_from_isolines_directory}/Synthetic_{year}yr_Bivariate_from_Trivariate_{number_of_events_per_year}.csv')
        for i in range(len(target_df)):
            target_center_variable_peak = target_df.loc[i,center_variable_synthetic_col]  

            # Sample and scale
            idx, probs = sample_events(historical_peaks, target_center_variable_peak, power=1.5, seed=42)
            print(f"Historical Peak: {historical_peaks[idx]}, Generated Peak: {target_center_variable_peak}, sampling probability: {probs[idx]:.3f}")
            historical_curves = df_timeseries_pot[df_timeseries_pot['event_time'] == df_pot['time'].iloc[idx]]
            selected_curve = historical_curves
            scaled_curve = scale_time_series(selected_curve[center_variable_dataset_col_name], target_center_variable_peak, curve_type="NTR")

            center_variable_scale_factor = target_center_variable_peak / selected_curve[center_variable_dataset_col_name].max()
            
            df_pot_center_variable_scaled = pd.DataFrame({
                'Time': pd.to_datetime(selected_curve[pot_timeseries_time].values),
                center_variable_output_col: scaled_curve,
                center_variable_output_scale_factor_col: center_variable_scale_factor
            })
            df_pot_center_variable_scaled['Time'] = pd.to_datetime(df_pot_center_variable_scaled['Time'])
    
            df_pot_center_variable_scaled = df_pot_center_variable_scaled.sort_values(by='Time')
            df_pot_center_variable_scaled.to_csv(center_variable_output_path_station_1, index=False)


    # ======================================================
    # Station 2
    # ======================================================
    df_original_timeseries = pd.read_csv(center_variable_dataset_path_station_2)
    df_original_timeseries['Time'] = pd.to_datetime(df_original_timeseries[center_variable_dataset_time], utc=True).dt.tz_localize(None)
    df_original_timeseries = df_original_timeseries.rename(columns={center_variable_dataset_col_name:'hourly_discharge_m3s'})
    df_original_timeseries = df_original_timeseries[['Time','hourly_discharge_m3s']]

    for year in years:
        target_df = pd.read_csv(rf'{synthetic_samples_from_isolines_directory}/Synthetic_{year}yr_Bivariate_from_Trivariate_{number_of_events_per_year}.csv')
        for i in range(len(target_df)):
            target_center_variable_peak = target_df.loc[i,center_variable_synthetic_col]  

            # Sample and scale
            idx, probs = sample_events(historical_peaks, target_center_variable_peak, power=1.5, seed=42)
            print(f"Historical Peak: {historical_peaks[idx]}, Generated Peak: {target_center_variable_peak}, sampling probability: {probs[idx]:.3f}")
            historical_curves = df_timeseries_pot[df_timeseries_pot['event_time'] == df_pot['time'].iloc[idx]]
            selected_curve = historical_curves
            scaled_curve = scale_time_series(selected_curve[center_variable_dataset_col_name], target_center_variable_peak, curve_type="NTR")

            center_variable_scale_factor = target_center_variable_peak / selected_curve[center_variable_dataset_col_name].max()
            
            df_pot_center_variable_scaled = pd.DataFrame({
                'Time': pd.to_datetime(selected_curve[pot_timeseries_time].values),
                center_variable_output_col: scaled_curve,
                center_variable_output_scale_factor_col: center_variable_scale_factor
            })
            df_pot_center_variable_scaled['Time'] = pd.to_datetime(df_pot_center_variable_scaled['Time'])
    
            df_pot_center_variable_scaled = df_pot_center_variable_scaled.sort_values(by='Time')
            df_pot_center_variable_scaled.to_csv(center_variable_output_path_station_2, index=False)


In [ ]:
def scale_and_align_associated_variable_one_station(associated_variable_dataset_col_name, 
                            associated_variable_dataset_time_col, 
                            associated_variable_dataset_path, 
                            associated_variable_pot_col_name, 
                            pot_time,
                            pot_timeseries_event_time, 
                            pot_timeseries_time, 
                            center_variable_pot_path,
                            center_variable_pot_timeseries_path, 
                            number_of_events_per_year, 
                            synthetic_samples_from_isolines_directory, 
                            associated_variable_synthetic_col,
                            lag_time_associated_variable, 
                            associated_variable_output_path, 
                            associated_variable_output_col, 
                            associated_variable_output_scale_factor_col, 
                            associated_variable_output_lag_time_col,
                            years):
    """
    Generates scaled time series for an associated variable (single station).

    Parameters
    ----------
    Similar to center-variable function.

    Returns
    -------
    None

    Notes
    -----
    The associated variable is sampled from events that occurred around the center-variable POT events.
    However, the selected associated-variable event may be different from the selected center-variable event.

    Thus, the associated variable is conditioned on the center-variable event pool, but it is selected
    according to its own target synthetic peak.
    """
    
    df_pot = pd.read_csv(center_variable_pot_path)
    df_pot['time'] = pd.to_datetime(df_pot[pot_time])
    historical_peaks = df_pot[associated_variable_pot_col_name].values  

    df_timeseries_pot = pd.read_csv(center_variable_pot_timeseries_path)
    df_timeseries_pot['event_time'] = pd.to_datetime(df_timeseries_pot[pot_timeseries_event_time])
    df_timeseries_pot = df_timeseries_pot.sort_values(by=['event_time'])


    df_pot_associated_variable = df_pot.copy()
    cutoff = pd.Timestamp('1978-12-31 00:00:00') ######## if your dataset has limited data before a specific date #############
    df_pot_associated_variable = df_pot_associated_variable[df_pot_associated_variable['time'] > cutoff]

    historical_associated_variable_peaks = df_pot_associated_variable[associated_variable_pot_col_name].values 

    associated_variable_lag_pool = df_timeseries_pot[lag_time_associated_variable].dropna().to_numpy()
    associated_variable_lag_pool = associated_variable_lag_pool[associated_variable_lag_pool < 120] ### The lag time should be less than 5 days
    rng = np.random.default_rng(42)  # set seed for reproducibility

    df_original_timeseries_associated_variable = pd.read_csv(associated_variable_dataset_path)
    df_original_timeseries_associated_variable = df_original_timeseries_associated_variable[[associated_variable_dataset_time_col, associated_variable_dataset_col_name]]
    df_original_timeseries_associated_variable = df_original_timeseries_associated_variable.rename(columns={associated_variable_dataset_time_col:'Time'})
    df_original_timeseries_associated_variable['Time'] = pd.to_datetime(df_original_timeseries_associated_variable['Time'])
    
    for year in years:
        target_df = pd.read_csv(rf'{synthetic_samples_from_isolines_directory}/Synthetic_{year}yr_Bivariate_from_Trivariate_{number_of_events_per_year}.csv')
        for i in range(len(target_df)):
            target_associated_variable_peak = target_df.loc[i,associated_variable_synthetic_col]  
            # Sample and scale
            idx, probs = sample_events(historical_associated_variable_peaks, target_associated_variable_peak, power=1.8, seed=42)
            print(f"Historical Peak: {historical_associated_variable_peaks[idx]}, Generated Peak: {target_associated_variable_peak}, sampling probability: {probs[idx]:.3f}")
            historical_associated_variable_curves = df_timeseries_pot[df_timeseries_pot['event_time'] == df_pot_associated_variable['time'].iloc[idx]]
            if len(historical_associated_variable_curves) > 0:
                selected_curve = historical_associated_variable_curves
                scaled_associated_variable_array, associated_variable_scale_factor = scale_time_series(selected_curve[associated_variable_dataset_col_name], target_associated_variable_peak)

                associated_variable_scale_factor = target_associated_variable_peak / selected_curve[associated_variable_dataset_col_name].max()

                sampled_associated_variable_lag = float(rng.choice(associated_variable_lag_pool)) if associated_variable_lag_pool.size > 0 else np.nan

                df_pot_associated_variable_scaled = pd.DataFrame({
                    'Time': pd.to_datetime(selected_curve[pot_timeseries_time].values),
                    associated_variable_output_col: scaled_associated_variable_array,
                    associated_variable_output_scale_factor_col: associated_variable_scale_factor,
                    associated_variable_output_lag_time_col: sampled_associated_variable_lag
                })
                df_pot_associated_variable_scaled['Time'] = pd.to_datetime(df_pot_associated_variable_scaled['Time'])
                
                df_merged = pd.merge(df_pot_associated_variable_scaled,df_original_timeseries_associated_variable, on ='Time')#
                df_merged['Real RF (mm)'] = df_merged[associated_variable_dataset_col_name]*df_merged['RF Scale Factor']
                df_merged = df_merged.sort_values(by='Time')
                df_merged.to_csv(associated_variable_output_path, index=False)

In [ ]:
def scale_and_align_associated_variable_two_stations(associated_variable_dataset_col_name,
                            associated_variable_dataset_time_col,
                            associated_variable_dataset_path_station_1, 
                            associated_variable_dataset_path_station_2,
                            associated_variable_pot_col_name, 
                            pot_time,
                            pot_timeseries_event_time, 
                            pot_timeseries_time, 
                            number_of_events_per_year, 
                            center_variable_pot_path,
                            center_variable_pot_timeseries_path, 
                            synthetic_samples_from_isolines_directory, 
                            associated_variable_synthetic_col, 
                            lag_time_associated_variable,
                            associated_variable_output_path_station_1, 
                            associated_variable_output_path_station_2, 
                            associated_variable_output_col, 
                            associated_variable_output_scale_factor_col,
                            associated_variable_output_lag_time_col,
                            years):
    """
    Generates scaled time series for an associated variable (single station).

    Parameters
    ----------
    Similar to center-variable function.

    Returns
    -------
    None

    Notes
    -----
    The associated variable is sampled from events that occurred around the center-variable POT events.
    However, the selected associated-variable event may be different from the selected center-variable event.

    Thus, the associated variable is conditioned on the center-variable event pool, but it is selected
    according to its own target synthetic peak.
    """

    df_pot = pd.read_csv(center_variable_pot_path)
    df_pot['time'] = pd.to_datetime(df_pot[pot_time])
    historical_peaks = df_pot[associated_variable_pot_col_name].values  

    df_timeseries_pot = pd.read_csv(center_variable_pot_timeseries_path)
    df_timeseries_pot['event_time'] = pd.to_datetime(df_timeseries_pot[pot_timeseries_event_time])
    df_timeseries_pot = df_timeseries_pot.sort_values(by=['event_time'])

    # ======================================================
    # Station 1
    # ======================================================
    df_pot_associated_variable = df_pot.copy()
    cutoff = pd.Timestamp('1987-11-10 23:59:59') ######## if your dataset has limited data before a specific date #############
    df_pot_associated_variable = df_pot_associated_variable[df_pot_associated_variable['time'] > cutoff]

    historical_associated_variable_peaks = df_pot_associated_variable[associated_variable_pot_col_name].values 

    associated_variable_lag_pool = df_timeseries_pot[lag_time_associated_variable].dropna().to_numpy()
    associated_variable_lag_pool = associated_variable_lag_pool[associated_variable_lag_pool < 120] ### The lag time should be less than 5 days
    rng = np.random.default_rng(42)  # set seed for reproducibility

    df_original_timeseries_associated_variable = pd.read_csv(associated_variable_dataset_path_station_1)
    df_original_timeseries_associated_variable = df_original_timeseries_associated_variable[[associated_variable_dataset_time_col, associated_variable_dataset_col_name]]
    df_original_timeseries_associated_variable = df_original_timeseries_associated_variable.rename(columns={associated_variable_dataset_time_col:'Time', associated_variable_dataset_col_name: 'hourly_discharge_m3s'})
    df_original_timeseries_associated_variable['Time'] = pd.to_datetime(df_original_timeseries_associated_variable['Time'])
    df_original_timeseries_associated_variable = df_original_timeseries_associated_variable[['Time','hourly_discharge_m3s']]
    
    for year in years:
        target_df = pd.read_csv(rf'{synthetic_samples_from_isolines_directory}/Synthetic_{year}yr_Bivariate_from_Trivariate_{number_of_events_per_year}.csv')
        for i in range(len(target_df)):
            target_associated_variable_peak = target_df.loc[i,associated_variable_synthetic_col]  
            # Sample and scale
            idx, probs = sample_events(historical_associated_variable_peaks, target_associated_variable_peak, power=1.8, seed=42)
            print(f"Historical Peak: {historical_associated_variable_peaks[idx]}, Generated Peak: {target_associated_variable_peak}, sampling probability: {probs[idx]:.3f}")
            historical_associated_variable_curves = df_timeseries_pot[df_timeseries_pot['event_time'] == df_pot_associated_variable['time'].iloc[idx]]
            if len(historical_associated_variable_curves) > 0:
                selected_curve = historical_associated_variable_curves
                scaled_associated_variable_array, associated_variable_scale_factor = scale_time_series(selected_curve[associated_variable_dataset_col_name], target_associated_variable_peak)

                associated_variable_scale_factor = target_associated_variable_peak / selected_curve[associated_variable_dataset_col_name].max()

                sampled_associated_variable_lag = float(rng.choice(associated_variable_lag_pool)) if associated_variable_lag_pool.size > 0 else np.nan

                df_pot_associated_variable_scaled = pd.DataFrame({
                    'Time': pd.to_datetime(selected_curve[pot_timeseries_time].values),
                    associated_variable_output_col: scaled_associated_variable_array,
                    associated_variable_output_scale_factor_col: associated_variable_scale_factor,
                    associated_variable_output_lag_time_col: sampled_associated_variable_lag
                })
                df_pot_associated_variable_scaled['Time'] = pd.to_datetime(df_pot_associated_variable_scaled['Time'])
                
                df_merged = pd.merge(df_pot_associated_variable_scaled,df_original_timeseries_associated_variable, on ='Time')#
                df_merged = df_merged.rename(columns={'hourly_discharge_m3s':'RD_Original_Hourly'})
                df_merged['RD_Scaled_Hourly'] = df_merged['RD_Original_Hourly'] * df_merged[associated_variable_output_scale_factor_col]
                df_merged = df_merged.sort_values(by='Time')
                # ensure proper datetime index, sorted, and numeric
                df_merged = df_merged.sort_values('Time').drop_duplicates('Time', keep='last')
                df_merged['Time'] = pd.to_datetime(df_merged['Time'])
                df_merged['RD_Scaled_Hourly'] = pd.to_numeric(df_merged['RD_Scaled_Hourly'], errors='coerce')
                
                # set index for time interpolation
                df_merged = df_merged.set_index('Time')

                df_merged['RD_Scaled_Hourly'] = df_merged['RD_Scaled_Hourly'].interpolate(
                    method='time'
                )

                df_merged = df_merged.reset_index()
                df_merged.to_csv(associated_variable_output_path_station_1, index=False)

    # ======================================================
    # Station 2
    # ======================================================
    df_pot_associated_variable = df_pot.copy()
    cutoff = pd.Timestamp('1993-11-10 23:59:59') ######## if your dataset has limited data before a specific date #############
    df_pot_associated_variable = df_pot_associated_variable[df_pot_associated_variable['time'] > cutoff]

    historical_associated_variable_peaks = df_pot_associated_variable[associated_variable_pot_col_name].values 

    associated_variable_lag_pool = df_timeseries_pot[lag_time_associated_variable].dropna().to_numpy()
    associated_variable_lag_pool = associated_variable_lag_pool[associated_variable_lag_pool < 120] ### The lag time should be less than 5 days
    rng = np.random.default_rng(42)  # set seed for reproducibility

    df_original_timeseries_associated_variable = pd.read_csv(associated_variable_dataset_path_station_2)
    df_original_timeseries_associated_variable = df_original_timeseries_associated_variable[[associated_variable_dataset_time_col, associated_variable_dataset_col_name]]
    df_original_timeseries_associated_variable = df_original_timeseries_associated_variable.rename(columns={associated_variable_dataset_time_col:'Time', associated_variable_dataset_col_name: 'hourly_discharge_m3s'})
    df_original_timeseries_associated_variable['Time'] = pd.to_datetime(df_original_timeseries_associated_variable['Time'])
    df_original_timeseries_associated_variable = df_original_timeseries_associated_variable[['Time','hourly_discharge_m3s']]
    
    for year in years:
        target_df = pd.read_csv(rf'{synthetic_samples_from_isolines_directory}/Synthetic_{year}yr_Bivariate_from_Trivariate_{number_of_events_per_year}.csv')
        for i in range(len(target_df)):
            target_associated_variable_peak = target_df.loc[i,associated_variable_synthetic_col]  
            # Sample and scale
            idx, probs = sample_events(historical_associated_variable_peaks, target_associated_variable_peak, power=1.8, seed=42)
            print(f"Historical Peak: {historical_associated_variable_peaks[idx]}, Generated Peak: {target_associated_variable_peak}, sampling probability: {probs[idx]:.3f}")
            historical_associated_variable_curves = df_timeseries_pot[df_timeseries_pot['event_time'] == df_pot_associated_variable['time'].iloc[idx]]
            if len(historical_associated_variable_curves) > 0:
                selected_curve = historical_associated_variable_curves
                scaled_associated_variable_array, associated_variable_scale_factor = scale_time_series(selected_curve[associated_variable_dataset_col_name], target_associated_variable_peak)

                associated_variable_scale_factor = target_associated_variable_peak / selected_curve[associated_variable_dataset_col_name].max()

                sampled_associated_variable_lag = float(rng.choice(associated_variable_lag_pool)) if associated_variable_lag_pool.size > 0 else np.nan

                df_pot_associated_variable_scaled = pd.DataFrame({
                    'Time': pd.to_datetime(selected_curve[pot_timeseries_time].values),
                    associated_variable_output_col: scaled_associated_variable_array,
                    associated_variable_output_scale_factor_col: associated_variable_scale_factor,
                    associated_variable_output_lag_time_col: sampled_associated_variable_lag
                })
                df_pot_associated_variable_scaled['Time'] = pd.to_datetime(df_pot_associated_variable_scaled['Time'])
                
                df_merged = pd.merge(df_pot_associated_variable_scaled,df_original_timeseries_associated_variable, on ='Time')#
                df_merged = df_merged.rename(columns={'hourly_discharge_m3s':'RD_Original_Hourly'})
                df_merged['RD_Scaled_Hourly'] = df_merged['RD_Original_Hourly'] * df_merged[associated_variable_output_scale_factor_col]
                df_merged = df_merged.sort_values(by='Time')
                # ensure proper datetime index, sorted, and numeric
                df_merged = df_merged.sort_values('Time').drop_duplicates('Time', keep='last')
                df_merged['Time'] = pd.to_datetime(df_merged['Time'])
                df_merged['RD_Scaled_Hourly'] = pd.to_numeric(df_merged['RD_Scaled_Hourly'], errors='coerce')
                
                # set index for time interpolation
                df_merged = df_merged.set_index('Time')

                df_merged['RD_Scaled_Hourly'] = df_merged['RD_Scaled_Hourly'].interpolate(
                    method='time'
                )

                df_merged = df_merged.reset_index()
                df_merged.to_csv(associated_variable_output_path_station_2, index=False)


In [ ]:
def merge_files(timeseries_center_variable_path, timeseries_center_variable_col, 
                timeseries_associated_variable_1_path, timeseries_associated_variable_1_col, timeseries_associated_variable_1_lag_time, timeseries_associated_variable_1_scale_factor,
                timeseries_associated_variable_2_path, timeseries_associated_variable_2_col, timeseries_associated_variable_2_lag_time, timeseries_associated_variable_2_scale_factor,
                output_path,
                years):

    """
    Aligns and merges three time-series variables (center variable and two associated variables)
    into a single event-based dataset using a relative-time framework centered on the peak of
    the center variable.

    Parameters
    ----------
    timeseries_center_variable_path : str
        Path to the time series file of the center variable (e.g., NTR or RD).

    timeseries_center_variable_col : str
        Column name of the center variable values in the dataset.

    timeseries_associated_variable_1_path : str
        Path to the time series file of the first associated variable (e.g., RF or NTR).

    timeseries_associated_variable_1_col : str
        Column name of the first associated variable.

    timeseries_associated_variable_1_lag_time : str
        Column containing lag values (in hours) for the first associated variable.

    timeseries_associated_variable_1_scale_factor : str
        Column containing scaling factors for the first associated variable.

    timeseries_associated_variable_2_path : str
        Path to the time series file of the second associated variable (e.g., RD).

    timeseries_associated_variable_2_col : str
        Column name of the second associated variable.

    timeseries_associated_variable_2_lag_time : str
        Column containing lag values (in hours) for the second associated variable.

    timeseries_associated_variable_2_scale_factor : str
        Column containing scaling factors for the second associated variable.

    output_path : str
        Path where the merged time series file will be saved.

    years : list of str
        List of return periods (e.g., ['1','5','10','20','50','100']).

    Returns
    -------
    None

    Workflow
    --------
    For each return period and event:

    1. Read the time series of the center variable and both associated variables.

    2. Identify the reference (peak position) for each variable by assuming the midpoint
       of the time series corresponds to the peak.

    3. Compute lag values for associated variables:
       - Associated variables are shifted in time relative to the center variable
         using their respective lag values.

    4. Construct a relative-time axis (in hours):
       - Center variable is anchored at rel_hour = 0.
       - Associated variables are shifted according to their lag values.

    5. Convert each time series into a relative-time representation:
       rel_hour = (index - peak_index) + lag

    6. Aggregate duplicate relative-hour values by averaging the variable values.

    7. Merge all variables on the relative time axis:
       - Center variable acts as the temporal reference.
       - Associated variables are aligned relative to it.

    8. Reconstruct an absolute time axis using the center variable peak time.

    9. Remove incomplete rows to ensure all variables are present.

    10. Save the merged dataset.

    Physical Interpretation
    -----------------------
    - The center variable defines the timing of the event (reference storm).
    - Associated variables are aligned relative to the center variable using lag times.
    - The resulting dataset represents a physically consistent multi-variable event.

    Important Assumptions
    ---------------------
    - The midpoint of each time series represents the peak of the event (we selected this way
    when we got the POT timeseries).
    - Lag values correctly represent temporal offsets between variables.
    - All input time series correspond to the same synthetic event.

    Notes
    -----
    This function does not perform event selection or scaling; it assumes that
    the input time series have already been generated and scaled appropriately.
    """

    for year in years:
        for i in range(100):

            # ======================================================
            # 1. Center variable
            # ======================================================
            df_center = pd.read_csv(timeseries_center_variable_path)
            df_center = df_center[['Time', timeseries_center_variable_col]]

            n_center = len(df_center)
            i0_center = (n_center - 1) // 2

            # ======================================================
            # 2. Associated variable 1
            # ======================================================
            df_assoc1 = pd.read_csv(timeseries_associated_variable_1_path)
            df_assoc1 = df_assoc1[
                ['Time',
                 timeseries_associated_variable_1_col,
                 timeseries_associated_variable_1_lag_time,
                 timeseries_associated_variable_1_scale_factor]
            ]

            lag_hour_assoc1 = df_assoc1[timeseries_associated_variable_1_lag_time].mean()
            n_assoc1 = len(df_assoc1)
            i0_assoc1 = (n_assoc1 - 1) // 2

            # ======================================================
            # 3. Associated variable 2
            # ======================================================
            df_assoc2 = pd.read_csv(timeseries_associated_variable_2_path)
            df_assoc2 = df_assoc2[
                ['Time',
                 timeseries_associated_variable_2_col,
                 timeseries_associated_variable_2_lag_time,
                 timeseries_associated_variable_2_scale_factor]
            ]

            lag_hour_assoc2 = df_assoc2[timeseries_associated_variable_2_lag_time].mean()
            assoc2_scale_factor = df_assoc2[timeseries_associated_variable_2_scale_factor].mean()

            n_assoc2 = len(df_assoc2)
            i0_assoc2 = (n_assoc2 - 1) // 2

            # ======================================================
            # Convert lags
            # ======================================================
            lag_hour_assoc1 = float(pd.to_numeric(lag_hour_assoc1, errors='coerce'))
            lag_hour_assoc2 = float(pd.to_numeric(lag_hour_assoc2, errors='coerce'))

            if not np.isfinite(lag_hour_assoc1):
                lag_hour_assoc1 = 0.0
            if not np.isfinite(lag_hour_assoc2):
                lag_hour_assoc2 = 0.0

            # ======================================================
            # Relative time axes
            # ======================================================
            rel_center = (np.arange(len(df_center)) - i0_center).astype(int)

            rel_assoc1 = (
                (np.arange(len(df_assoc1)) - i0_assoc1) +
                int(round(lag_hour_assoc1))
            ).astype(int)

            rel_assoc2 = (
                (np.arange(len(df_assoc2)) - i0_assoc2) +
                int(round(lag_hour_assoc2))
            ).astype(int)

            # ======================================================
            # Build frames
            # ======================================================
            center_frame = pd.DataFrame({
                "rel_hour": rel_center,
                timeseries_center_variable_col: df_center[timeseries_center_variable_col].values,
                "Center_Time": pd.to_datetime(df_center["Time"], errors="coerce")
            })

            assoc1_frame = pd.DataFrame({
                "rel_hour": rel_assoc1,
                timeseries_associated_variable_1_col: df_assoc1[timeseries_associated_variable_1_col].values,
                "Assoc1_Time": pd.to_datetime(df_assoc1["Time"], errors="coerce")
            })

            assoc2_frame = pd.DataFrame({
                "rel_hour": rel_assoc2,
                timeseries_associated_variable_2_col: df_assoc2[timeseries_associated_variable_2_col].values,
                "Assoc2_Time": pd.to_datetime(df_assoc2["Time"], errors="coerce")
            })

            # ======================================================
            # Aggregate duplicates
            # ======================================================
            center_frame = center_frame.groupby("rel_hour", as_index=False).agg({
                timeseries_center_variable_col: "mean",
                "Center_Time": "first"
            })

            assoc1_frame = assoc1_frame.groupby("rel_hour", as_index=False).agg({
                timeseries_associated_variable_1_col: "mean",
                "Assoc1_Time": "first"
            })

            assoc2_frame = assoc2_frame.groupby("rel_hour", as_index=False).agg({
                timeseries_associated_variable_2_col: "mean",
                "Assoc2_Time": "first"
            })

            # ======================================================
            # Merge (center = anchor)
            # ======================================================
            merged = (
                center_frame
                .merge(assoc1_frame, on="rel_hour", how="outer")
                .merge(assoc2_frame, on="rel_hour", how="outer")
                .sort_values("rel_hour")
                .reset_index(drop=True)
            )

            # ======================================================
            # Anchor flags
            # ======================================================
            merged["is_anchor_center"] = (merged["rel_hour"] == 0)
            merged["is_anchor_assoc1"] = (merged["rel_hour"] == int(round(lag_hour_assoc1)))
            merged["is_anchor_assoc2"] = (merged["rel_hour"] == int(round(lag_hour_assoc2)))

            # ======================================================
            # Reconstruct absolute time
            # ======================================================
            t_anchor = pd.to_datetime(df_center.loc[i0_center, "Time"], errors="coerce")

            if pd.notna(t_anchor):
                merged["Time"] = t_anchor + pd.to_timedelta(merged["rel_hour"], unit="h")

            # ======================================================
            # Clean + metadata
            # ======================================================
            merged = merged.dropna(subset=[
                timeseries_center_variable_col,
                timeseries_associated_variable_1_col,
                timeseries_associated_variable_2_col
            ])

            merged = merged.drop(columns=["rel_hour"], errors="ignore")

            merged["Lag_Hour_Assoc1"] = lag_hour_assoc1
            merged["Lag_Hour_Assoc2"] = lag_hour_assoc2
            merged["Assoc2 Scale Factor"] = assoc2_scale_factor

            # ======================================================
            # Save
            # ======================================================
            merged.to_csv(output_path, index=False)